In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tigramite import data_processing as pp
from tigramite import plotting as tp
from tigramite.pcmci import PCMCI

def create_simple_causal_graphs():
    """
    Create and plot simple causal graph structures.
    """
    # Define graph structures
    graphs = {
        'X->Z<-Y': np.array([
            [[0, 0], [0, 0]],   # X
            [[0, 0], [0, 0]],   # Y
            [[1, 0], [1, 0]]    # Z
        ]),
        'X->Z->Y': np.array([
            [[0, 0], [0, 0]],   # X
            [[0, 0], [1, 0]],   # Y
            [[1, 0], [0, 0]]    # Z
        ]),
        'X<-Z->Y': np.array([
            [[0, 0], [1, 0]],   # X
            [[0, 0], [1, 0]],   # Y
            [[0, 0], [0, 0]]    # Z
        ])
    }
    
    # Create plots
    for name, graph in graphs.items():
        plt.figure(figsize=(6, 4))
        # Create a value matrix of the same shape filled with 1s
        val_matrix = np.ones_like(graph, dtype=float)
        tp.plot_graph(val_matrix, graph, var_names=['X', 'Y', 'Z'])
        plt.title(f'Causal Graph: {name}')
        plt.tight_layout()
        plt.savefig(f'{name}_causal_graph.png')
        plt.close()

def create_time_series_graph(auto_coeff=0.8, cross_coeff=0.5):
    """
    Create a time series graph plot with color-coded auto and cross links.
    """
    # Define the links structure
    links = {
        0: [((0, -1), auto_coeff, lambda x: x), 
            ((2, -1), cross_coeff, lambda x: x)],
        1: [((1, -1), auto_coeff, lambda x: x), 
            ((0, -1), cross_coeff, lambda x: x)],
        2: [((2, -1), auto_coeff, lambda x: x)],       
        3: [((3, -1), auto_coeff, lambda x: x),
            ((0, -1), cross_coeff, lambda x: x), 
            ((1, -1), cross_coeff, lambda x: x), 
            ((2, -2), cross_coeff, lambda x: x)]
    }
    
    # Convert links to graph
    n_vars = max(links.keys()) + 1
    max_lag = max(max(abs(lag) for (_, lag), _, _ in var_links) for var_links in links.values()) + 1
    graph = np.zeros((n_vars, n_vars, max_lag), dtype=int)
    val_matrix = np.zeros_like(graph, dtype=float)
    
    for var, var_links in links.items():
        for (parent_var, parent_lag), coeff, _ in var_links:
            if parent_var != var:
                # Cross-link
                graph[var, parent_var, abs(parent_lag)] = 1
                val_matrix[var, parent_var, abs(parent_lag)] = coeff
            else:
                # Auto-link
                graph[var, parent_var, abs(parent_lag)] = 2
                val_matrix[var, parent_var, abs(parent_lag)] = coeff
    
    # Var names
    var_names = [f'X{i+1}' for i in range(n_vars)]
    
    # Create plot
    plt.figure(figsize=(10, 6))
    tp.plot_time_series_graph(
        graph=graph,
        var_names=var_names,
        link_colorbar_label='Link Strength'
    )
    plt.title('Time Series Causal Graph')
    plt.tight_layout()
    plt.savefig('time_series_causal_graph.png')
    plt.close()

def create_dag_plot(auto_coeff=0.8, cross_coeff=0.5):
    """
    Create a standard DAG plot for the time series structure.
    """
    # Define the links structure
    links = {
        0: [((0, -1), auto_coeff, lambda x: x), 
            ((2, -1), cross_coeff, lambda x: x)],
        1: [((1, -1), auto_coeff, lambda x: x), 
            ((0, -1), cross_coeff, lambda x: x)],
        2: [((2, -1), auto_coeff, lambda x: x)],       
        3: [((3, -1), auto_coeff, lambda x: x),
            ((0, -1), cross_coeff, lambda x: x), 
            ((1, -1), cross_coeff, lambda x: x), 
            ((2, -2), cross_coeff, lambda x: x)]
    }
    
    # Convert links to graph
    n_vars = max(links.keys()) + 1
    graph = np.zeros((n_vars, n_vars), dtype=int)
    val_matrix = np.zeros_like(graph, dtype=float)
    
    for var, var_links in links.items():
        for (parent_var, _), coeff, _ in var_links:
            if parent_var != var:
                graph[var, parent_var] = 1
                val_matrix[var, parent_var] = coeff
    
    # Var names
    var_names = [f'X{i+1}' for i in range(n_vars)]
    
    # Create plot
    plt.figure(figsize=(8, 6))
    tp.plot_graph(
        graph=graph, 
        var_names=var_names
    )
    plt.title('DAG Causal Graph')
    plt.tight_layout()
    plt.savefig('dag_causal_graph.png')
    plt.close()

def main():
    # Generate all plots
    #create_simple_causal_graphs()
    create_time_series_graph()
    create_dag_plot()
    print("All plots have been generated successfully!")

if __name__ == '__main__':
    main()